In [2]:
from datasets import load_dataset

/home/daniel/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
ds = load_dataset("allenai/RLVR-IFeval", split="train")
ds

Generating train split: 100%|██████████| 14973/14973 [00:00<00:00, 150522.54 examples/s]


Dataset({
    features: ['messages', 'ground_truth', 'dataset', 'constraint_type', 'constraint'],
    num_rows: 14973
})

In [30]:
ds_debug = ds.select([
    ds.filter(lambda x: x["constraint_type"] == ct)[0] 
    for ct in set(ds["constraint_type"], num_proc=16)
])
ds_debug

Filter: 100%|██████████| 14973/14973 [00:00<00:00, 105050.54 examples/s]


TypeError: '>=' not supported between instances of 'dict' and 'int'

In [10]:
num_proc = 16
ds_up = ds.map(lambda x, idx: {"problem_id": f"ifeval_{idx}"}, with_indices=True, num_proc=num_proc)
ds_up = ds_up.map(lambda x: {"prompt": x["messages"][0]["content"]}, num_proc=num_proc)

Map (num_proc=16): 100%|██████████| 14973/14973 [00:00<00:00, 64548.58 examples/s]


In [11]:
ds_up = ds_up.map(lambda x: {"verification_info": repr({"ground_truth": x["ground_truth"]})}, num_proc=num_proc)

Map (num_proc=16): 100%|██████████| 14973/14973 [00:00<00:00, 61275.61 examples/s]


In [13]:
print(ds_up[0]["verification_info"])

{'ground_truth': '{"func_name": "validate_lowercase", "N": null, "quantifier": null, "end_phrase": null, "keyword_list": null, "word": null, "forbidden_words": null, "letter": null, "i": null, "first_word": null, "postscript_marker": null, "options": null, "section_splitter": null, "original_prompt": null}'}


In [14]:
ds_up = ds_up.map(lambda x: {"task_type": "ifeval"}, num_proc=num_proc)

Map (num_proc=16): 100%|██████████| 14973/14973 [00:00<00:00, 26961.36 examples/s]


In [23]:
ds_up = ds_up.map(lambda x: {
    "metadata": repr({
        "constraint_type": x["constraint_type"],
        "constraint": x["constraint"]
    })
}, num_proc=num_proc)

Map (num_proc=16):   0%|          | 0/14973 [00:00<?, ? examples/s]

Map (num_proc=16): 100%|██████████| 14973/14973 [00:00<00:00, 54713.94 examples/s]


In [24]:
ds_up = ds_up.map(lambda x: {"source": "allenai/RLVR-IFeval"}, num_proc=num_proc)

Map (num_proc=16): 100%|██████████| 14973/14973 [00:00<00:00, 56578.63 examples/s]


In [25]:
ds_sub = ds_up.select_columns(["problem_id", "source", "task_type", "prompt", "verification_info", "metadata"])

In [26]:
ds_sub.push_to_hub("rasdani/ifeval-genesys")

Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.31s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/rasdani/ifeval-genesys/commit/0f936e43d12a3323766fd0fe6f1bcee21229d263', commit_message='Upload dataset', commit_description='', oid='0f936e43d12a3323766fd0fe6f1bcee21229d263', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/rasdani/ifeval-genesys', endpoint='https://huggingface.co', repo_type='dataset', repo_id='rasdani/ifeval-genesys'), pr_revision=None, pr_num=None)

In [28]:
ds_verified = load_dataset("json", data_files="../output/ifeval_30/output_verified.jsonl")
ds_verified

Generating train split: 30 examples [00:00, 5691.57 examples/s]


DatasetDict({
    train: Dataset({
        features: ['problem_id', 'source', 'task_type', 'prompt', 'verification_info', 'metadata', 'llm_response', 'score', 'verification_result_info'],
        num_rows: 30
    })
})

In [29]:
ds_verified.push_to_hub("rasdani/ifeval-genesys-verified")

Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.29s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/rasdani/ifeval-genesys-verified/commit/912b0a34cf99334f559879d0ff4ac8b24473dabc', commit_message='Upload dataset', commit_description='', oid='912b0a34cf99334f559879d0ff4ac8b24473dabc', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/rasdani/ifeval-genesys-verified', endpoint='https://huggingface.co', repo_type='dataset', repo_id='rasdani/ifeval-genesys-verified'), pr_revision=None, pr_num=None)